# 🚀 End-to-End LLM Fine-Tuning with Unsloth — Roman Urdu Comments ke Saath
> **Maqsad:** Unsloth library se TinyLlama model ko 3 stages mein fine-tune karna, **sequence mein**:
> - **Stage 1 (Non-Instruction FT):** PDF se pharma domain language sikhao
> - **Stage 2 (Instruction FT / SFT):** Q&A format sikhao — user instructions follow karna
> - **Stage 3 (Preference / DPO):** Behtar responses choose karna sikhao
>
> **Is version mein kya alag hai?**
> Pehle wale notebook mein teenon stages ek hi "common" function call karte the
> (`load_unsloth_model_with_lora`, `train_and_measure`, `save_adapter_and_merge`, `generate_answer`).
> Isse sequence samajhna mushkil ho jata tha — kyunke asal training code kahin chhupa hota tha.
>
> **Ab har stage ka code bilkul separate aur poora likha hai** — model load karna, training
> configure karna, train karna, test karna, save/merge karna — sab kuch us stage ke apne
> code cell ke andar, bina kisi shared "black box" function ke. Thodा code repeat hoga,
> lekin sequence padhna aasaan hoga.
>
> **Unsloth kyun?** Normal Hugging Face se 2x fast training, 60% kam GPU memory


## 🔧 Step 1: Zaroori Libraries Install Karna

Pehle woh sab tools install karte hain jo is notebook mein use honge.


In [ ]:
# ============================================================
# 1. Zaroori Libraries Install Karna
# ============================================================

# unsloth      → Unsloth ka main package — model loading, LoRA, aur training
#                Normal transformers se 2x fast hai aur 60% kam VRAM use karta hai
#                Ye automatically bitsandbytes, xformers, aur peft ko patch karta hai

# transformers → Hugging Face ka core library — model classes, tokenizer, Trainer
# ==4.56.2     → Fixed version — unsloth ke saath compatibility zaruri hai
#                Version mismatch se crashes aa sakte hain

# trl          → Transformer Reinforcement Learning library
# ==0.22.2     → SFTTrainer aur DPOTrainer yahan se aate hain
# --no-deps    → TRL ki dependencies dobara install mat karo — unsloth ne already sab set kiya hai

# pymupdf      → PDF files se text nikalne ke liye (fitz library)
# datasets     → Hugging Face Dataset format — training data manage karta hai

!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets


## 📦 Step 2: Libraries Import Karna

Install ke baad, unhe code mein use karne ke liye import karna padta hai.


In [ ]:
# ============================================================
# 2. Imports — Sari Zaroori Libraries Ek Jagah Load Karna
# ============================================================

import os           # Operating system functions — file paths, folders banana
import re           # Regular expressions — text patterns dhundhne ke liye
import gc           # Garbage collection — memory free karne ke liye
import time         # Training time measure karne ke liye
import json
import unicodedata  # Unicode characters normalize karne ke liye (PDF cleaning)
import warnings     # Python warnings suppress karne ke liye
from typing import List, Dict, Any

warnings.filterwarnings("ignore")  # Unnecessary warnings chhupaao — clean output ke liye

import torch        # PyTorch — deep learning framework, GPU operations
import fitz         # PyMuPDF — PDF se text extract karne ki main library

from datasets import Dataset, load_dataset
# Dataset      → Single dataset object
# load_dataset → JSONL/CSV/Hub se dataset load karne ke liye

import unsloth  # Unsloth ka core module — yeh pehle import hona chahiye
from unsloth import FastLanguageModel, is_bfloat16_supported

from trl import SFTTrainer, SFTConfig
# SFTTrainer → Supervised Fine-Tuning trainer (Stage 1 aur Stage 2 ke liye)
# SFTConfig  → SFTTrainer ki settings class

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()           # DPO trainer ko Unsloth ke saath patch karo
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig
# DPOTrainer → Direct Preference Optimization trainer (Stage 3 ke liye)
# DPOConfig  → DPOTrainer ki settings class

# GPU zaruri hai — yeh notebook CPU pe nahi chalegi
assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))


## ⚙️ Step 3: File Paths aur Configuration Set Karna

Saari important settings aur file paths ek jagah rakhte hain — yeh sirf **values** hain
(functions nahi), is liye teenon stages mein inhe use karna sequence ko confuse nahi karta.


In [ ]:
# ============================================================
# 3. File Paths aur Configuration — Saari Settings Ek Jagah
# ============================================================

# ── Input File Paths ──────────────────────────────────────────────────────────
non_instruction_data_path = "/content/Metformin-Lipid-Therapy-Raw-Data.pdf"
# Stage 1 ke liye raw PDF — pharma domain ki textbook ya research paper

instruction_data_path = "/content/pharma_instruction_dataset.jsonl"
# Stage 2 ke liye instruction data — format: {"instruction": "...", "output": "..."}

preference_data_path = "/content/pharma_preference_dataset.jsonl"
# Stage 3 ke liye preference data — format: {"prompt": "...", "chosen": "...", "rejected": "..."}

for path in [non_instruction_data_path, instruction_data_path, preference_data_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")

# ── Base Model ────────────────────────────────────────────────────────────────
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

# ── Sequence Length ───────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 512

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

# ── Text Preprocessing ────────────────────────────────────────────────────────
MIN_CHARS_PER_PARAGRAPH = 80

# ── LoRA Parameters (Teenon Stages ke Liye Same) ──────────────────────────────
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

# ── Training Batch Settings (Teenon Stages ke Liye Same) ─────────────────────
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
WARMUP_STEPS = 5
LOGGING_STEPS = 1

# ── Stage-wise Max Steps (Demo Mode) ─────────────────────────────────────────
STAGE1_MAX_STEPS = 30
STAGE2_MAX_STEPS = 30
STAGE3_MAX_STEPS = 30

# ── Stage-wise Learning Rates ─────────────────────────────────────────────────
STAGE1_LR = 2e-4   # Domain adaptation ke liye thoda higher
STAGE2_LR = 1e-4   # Instruction format ke liye thoda lower
STAGE3_LR = 5e-5   # DPO ke liye sabse low

# ── DPO Specific ──────────────────────────────────────────────────────────────
DPO_BETA = 0.1

# ── Output Directories ────────────────────────────────────────────────────────
OUTPUT_ROOT = "/content/unsloth_pharma_merge_reload_outputs"

STAGE1_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage1_non_instruction_adapter"
STAGE1_MERGED_DIR  = f"{OUTPUT_ROOT}/stage1_non_instruction_merged_model"

STAGE2_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage2_instruction_adapter"
STAGE2_MERGED_DIR  = f"{OUTPUT_ROOT}/stage2_instruction_merged_model"

STAGE3_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage3_dpo_adapter"
FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/stage3_dpo_final_merged_model"

for path in [
    OUTPUT_ROOT,
    STAGE1_ADAPTER_DIR, STAGE1_MERGED_DIR,
    STAGE2_ADAPTER_DIR, STAGE2_MERGED_DIR,
    STAGE3_ADAPTER_DIR, FINAL_MERGED_DIR,
]:
    os.makedirs(path, exist_ok=True)

print("All directories created.")


## 📄 Step 4: PDF Se Text Nikalna — Stage 1 Data Preparation

Stage 1 ke liye raw pharma PDF se clean paragraphs nikalo. (Yeh functions sirf data-cleaning
ke liye hain, training logic ke liye nahi — is liye yahan rakhna sequence ko confuse nahi karta.)


In [ ]:
# ============================================================
# 4. PDF Se Text Extract Karne Ka Function — Stage 1 Data
# ============================================================

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    """PDF ki har page se text extract karo."""
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({"page": page_number, "text": text})
    return pages


def clean_pdf_text(text: str) -> str:
    """Raw PDF text se hyphenated breaks, page numbers, extra whitespace clean karo."""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "")
    text = text.replace("\ufeff", "")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()
        if paragraph:
            paragraphs.append(paragraph)

    return "\n\n".join(paragraphs)


def build_pdf_dataset(pdf_path: str) -> Dataset:
    """PDF path se seedha Hugging Face Dataset banao."""
    pages = extract_pdf_pages(pdf_path)
    records = []

    for page in pages:
        cleaned_text = clean_pdf_text(page["text"])
        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):
            paragraph = paragraph.strip()
            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:
                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))
    print("\nSample paragraph:\n", records[0]["text"][:700])

    return Dataset.from_list(records)


# PDF dataset banao — Stage 1 ke liye
stage1_dataset = build_pdf_dataset(non_instruction_data_path)


---
# 🏋️ STAGE 1 — NON-INSTRUCTION FINE-TUNING (Domain Pretraining)

**Yahan se Stage 1 ka poora code shuru hota hai — sab kuch isi cell mein, kisi shared
function ke bina:**
1. Base model + LoRA load karna
2. Training configure karna
3. Train karna (time/VRAM khud measure karte hain)
4. Test answer generate karna
5. Adapter save + base model ke saath merge karna


In [ ]:
# ============================================================
# STAGE 1 — NON-INSTRUCTION FINE-TUNING (poora code, separate)
# Maqsad: Model ko pharma domain ki bhasha sikhana (raw PDF paragraphs se)
# ============================================================

print("\n==============================")
print("STAGE 1: PDF RAW TEXT TRAINING")
print("==============================")

# ── 1.1 Base Model + Tokenizer Load Karo (4-bit) ──────────────────────────────
stage1_model, stage1_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,     # Original unsloth TinyLlama (koi previous training nahi)
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,                     # None = automatically best dtype choose karo
    load_in_4bit=True,              # 4-bit quantization — 8x kam VRAM
)

if stage1_tokenizer.pad_token is None:
    stage1_tokenizer.pad_token = stage1_tokenizer.eos_token

stage1_tokenizer.padding_side = "right"   # Causal LM ke liye standard

# ── 1.2 LoRA Adapters Lagao ────────────────────────────────────────────────────
stage1_model = FastLanguageModel.get_peft_model(
    stage1_model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
stage1_model.print_trainable_parameters()

FastLanguageModel.for_training(stage1_model)  # Training mode on

# ── 1.3 Stage 1 Training Configuration ────────────────────────────────────────
stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    max_steps=STAGE1_MAX_STEPS,   # Demo ke liye 30 steps — production mein 500-2000
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,  # 0.0002 — domain adaptation ke liye higher
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,

    packing=True,  # ← Stage 1 mein TRUE — chote raw paragraphs ko ek saath pack karo
                   # Stage 2 mein FALSE hoga, kyunke instruction boundary preserve karni hai

    seed=SEED,
)

# ── 1.4 Trainer Banao ──────────────────────────────────────────────────────────
stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=stage1_tokenizer,
    train_dataset=stage1_dataset,
    args=stage1_config,
)

# ── 1.5 Training Chalao — Time aur VRAM Khud Measure Karo ─────────────────────
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

stage1_start_time = time.time()
stage1_train_result = stage1_trainer.train()
torch.cuda.synchronize()

stage1_train_time = round(time.time() - stage1_start_time, 2)
stage1_peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
stage1_peak_reserved  = round(torch.cuda.max_memory_reserved()  / 1024**3, 3)

print("\nSTAGE 1 RESULTS")
print("Train time/sec:", stage1_train_time)
print("Peak allocated VRAM/GB:", stage1_peak_allocated)
print("Peak reserved VRAM/GB:", stage1_peak_reserved)

# ── 1.6 Stage 1 Ke Baad Test Karo ─────────────────────────────────────────────
FastLanguageModel.for_inference(stage1_model)

stage1_test_prompt = "### Instruction:\nExplain metformin in simple language.\n\n### Response:\n"
stage1_inputs = stage1_tokenizer(stage1_test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    stage1_output_ids = stage1_model.generate(
        **stage1_inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=stage1_tokenizer.eos_token_id,
        eos_token_id=stage1_tokenizer.eos_token_id,
    )

stage1_input_len = stage1_inputs["input_ids"].shape[-1]
stage1_answer = stage1_tokenizer.decode(
    stage1_output_ids[0][stage1_input_len:], skip_special_tokens=True
).strip()
print("\nStage 1 test answer:\n", stage1_answer)

# ── 1.7 Adapter Save Karo + Base Model Ke Saath Merge Karo ───────────────────
FastLanguageModel.for_training(stage1_model)  # Save/merge se pehle training mode mein wapas aao

print("\nSaving Stage 1 adapter...")
stage1_model.save_pretrained(STAGE1_ADAPTER_DIR)
stage1_tokenizer.save_pretrained(STAGE1_ADAPTER_DIR)
print("Stage 1 adapter saved to:", STAGE1_ADAPTER_DIR)

print("\nMerging Stage 1 adapter with base model...")
stage1_model.save_pretrained_merged(
    STAGE1_MERGED_DIR,
    stage1_tokenizer,
    save_method="merged_16bit",  # Merged model ko float16 mein save karo
)
print("Stage 1 merged model saved to:", STAGE1_MERGED_DIR)

# ── 1.8 Memory Free Karo — Stage 2 Ke Liye VRAM Chahiye ───────────────────────
del stage1_trainer
del stage1_model
gc.collect()
torch.cuda.empty_cache()
print("Stage 1 complete. Memory cleared for Stage 2.")


## 📋 Stage 2 Data — Instruction Dataset Load Karna

JSONL file se instruction-response pairs load karo aur format karo.


In [ ]:
# ============================================================
# STAGE 2 DATA: Instruction JSONL Load aur Format Karna
# ============================================================

print("\n==============================")
print("STAGE 2: INSTRUCTION DATA")
print("==============================")

instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train",
)

required_instruction_cols = {"instruction", "output"}
missing_cols = required_instruction_cols - set(instruction_dataset.column_names)
if missing_cols:
    raise ValueError(f"Instruction dataset missing columns: {missing_cols}")


def format_instruction_record(example):
    """Har instruction record ko training ke liye '### Instruction / ### Response' text mein convert karo."""
    instruction = str(example.get("instruction", "")).strip()
    input_text  = str(example.get("input", "")).strip()
    output      = str(example.get("output", "")).strip()

    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    return {"text": prompt + output}


stage2_dataset = instruction_dataset.map(format_instruction_record)

print("Instruction rows:", len(stage2_dataset))
print("\nSample instruction text:\n", stage2_dataset[0]["text"][:900])


---
# 🎯 STAGE 2 — INSTRUCTION FINE-TUNING (SFT)

**Stage 1 ke merged model se shuru karte hain.** Poora code yahan dobara likha hai —
Stage 1 jaisa hi structure hai, lekin har line yahan dikhti hai, kisi function ke andar
chhupi nahi hai. Farq dhyan se dekhna: `packing=False`, lower `learning_rate`, aur
model loading path `STAGE1_MERGED_DIR` se.


In [ ]:
# ============================================================
# STAGE 2 — INSTRUCTION FINE-TUNING (poora code, separate)
# Maqsad: Model ko Q&A format sikhana
# Starting point: Stage 1 ka trained merged model
# ============================================================

print("\n===============================================")
print("STAGE 2: LOAD STAGE 1 MERGED MODEL AND TRAIN")
print("===============================================")

# ── 2.1 Stage 1 Merged Model + Tokenizer Load Karo (4-bit) ────────────────────
stage2_model, stage2_tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE1_MERGED_DIR,   # Stage 1 ka trained merged model — progressive training
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if stage2_tokenizer.pad_token is None:
    stage2_tokenizer.pad_token = stage2_tokenizer.eos_token

stage2_tokenizer.padding_side = "right"

# ── 2.2 Fresh LoRA Adapters Lagao ──────────────────────────────────────────────
stage2_model = FastLanguageModel.get_peft_model(
    stage2_model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
stage2_model.print_trainable_parameters()

FastLanguageModel.for_training(stage2_model)

# ── 2.3 Stage 2 Training Configuration ─────────────────────────────────────────
stage2_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage2_logs",

    max_steps=STAGE2_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE2_LR,  # 0.0001 — Stage 1 se CHOTA learning rate
                               # Model pehle se trained hai, careful update chahiye

    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,

    packing=False,  # ← Stage 2 mein FALSE — instruction boundary preserve karni hai
                    # True hota toh alag Q&A pairs galat tarike se mix ho jaate

    seed=SEED,
)

# ── 2.4 Trainer Banao ──────────────────────────────────────────────────────────
stage2_trainer = SFTTrainer(
    model=stage2_model,
    processing_class=stage2_tokenizer,
    train_dataset=stage2_dataset,
    args=stage2_config,
)

# ── 2.5 Training Chalao — Time aur VRAM Khud Measure Karo ─────────────────────
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

stage2_start_time = time.time()
stage2_train_result = stage2_trainer.train()
torch.cuda.synchronize()

stage2_train_time = round(time.time() - stage2_start_time, 2)
stage2_peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
stage2_peak_reserved  = round(torch.cuda.max_memory_reserved()  / 1024**3, 3)

print("\nSTAGE 2 RESULTS")
print("Train time/sec:", stage2_train_time)
print("Peak allocated VRAM/GB:", stage2_peak_allocated)
print("Peak reserved VRAM/GB:", stage2_peak_reserved)

# ── 2.6 Stage 2 Ke Baad Test Karo ─────────────────────────────────────────────
FastLanguageModel.for_inference(stage2_model)

stage2_test_prompt = "### Instruction:\nExplain metformin in simple language.\n\n### Response:\n"
stage2_inputs = stage2_tokenizer(stage2_test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    stage2_output_ids = stage2_model.generate(
        **stage2_inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=stage2_tokenizer.eos_token_id,
        eos_token_id=stage2_tokenizer.eos_token_id,
    )

stage2_input_len = stage2_inputs["input_ids"].shape[-1]
stage2_answer = stage2_tokenizer.decode(
    stage2_output_ids[0][stage2_input_len:], skip_special_tokens=True
).strip()
print("\nStage 2 test answer:\n", stage2_answer)

# ── 2.7 Adapter Save Karo + Merge Karo ────────────────────────────────────────
FastLanguageModel.for_training(stage2_model)

print("\nSaving Stage 2 adapter...")
stage2_model.save_pretrained(STAGE2_ADAPTER_DIR)
stage2_tokenizer.save_pretrained(STAGE2_ADAPTER_DIR)
print("Stage 2 adapter saved to:", STAGE2_ADAPTER_DIR)

print("\nMerging Stage 2 adapter with base model...")
stage2_model.save_pretrained_merged(
    STAGE2_MERGED_DIR,
    stage2_tokenizer,
    save_method="merged_16bit",
)
print("Stage 2 merged model saved to:", STAGE2_MERGED_DIR)

# ── 2.8 Memory Free Karo — Stage 3 Ke Liye VRAM Chahiye ───────────────────────
del stage2_trainer
del stage2_model
gc.collect()
torch.cuda.empty_cache()
print("Stage 2 complete. Memory cleared for Stage 3.")


## 🏆 Stage 3 Data — Preference Dataset Load Karna

DPO ke liye chosen/rejected pairs load karo — model seekhega ke kaunsa jawab behtar hai.


In [ ]:
# ============================================================
# STAGE 3 DATA: Preference (DPO) JSONL Load Karna
# ============================================================

print("\n==============================")
print("STAGE 3: PREFERENCE DATA")
print("==============================")

preference_dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train",
)

required_preference_cols = {"prompt", "chosen", "rejected"}
missing_cols = required_preference_cols - set(preference_dataset.column_names)
if missing_cols:
    raise ValueError(f"Preference dataset missing columns: {missing_cols}")


def clean_preference_record(example):
    """Preference records se whitespace clean karo."""
    return {
        "prompt":   str(example["prompt"]).strip(),
        "chosen":   str(example["chosen"]).strip(),
        "rejected": str(example["rejected"]).strip(),
    }


stage3_dataset = preference_dataset.map(clean_preference_record)

print("Preference rows:", len(stage3_dataset))
print("\nSample preference record:\n", stage3_dataset[0])


---
# 🎖️ STAGE 3 — PREFERENCE TUNING (DPO)

**Stage 2 ke merged model se shuru karte hain — yeh akhri stage hai.** Poora DPO code
yahan dobara, separately likha hai. Dhyan dene wali baatein: `tokenizer.padding_side = "left"`
(DPO ke liye zaruri), `DPOConfig`/`DPOTrainer` ka istemal (SFT ki jagah), aur sab se chota
`learning_rate`.


In [ ]:
# ============================================================
# STAGE 3 — DPO PREFERENCE TUNING (poora code, separate)
# Maqsad: Model ko behtar responses choose karna sikhana
# Starting point: Stage 2 ka trained merged model
# ============================================================

print("\n==========================================")
print("STAGE 3: LOAD STAGE 2 MERGED MODEL AND DPO")
print("==========================================")

# ── 3.1 Stage 2 Merged Model + Tokenizer Load Karo (4-bit) ────────────────────
stage3_model, stage3_tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE2_MERGED_DIR,   # Stage 2 ka trained merged model
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if stage3_tokenizer.pad_token is None:
    stage3_tokenizer.pad_token = stage3_tokenizer.eos_token

# ── 3.2 Fresh LoRA Adapters Lagao ──────────────────────────────────────────────
stage3_model = FastLanguageModel.get_peft_model(
    stage3_model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
stage3_model.print_trainable_parameters()

FastLanguageModel.for_training(stage3_model)

# ── 3.3 DPO Ke Liye Left Padding ──────────────────────────────────────────────
stage3_tokenizer.padding_side = "left"
# DPO ko prompt aur response dono ko align karna hota hai — decoder-only models
# mein left padding se alignment better hoti hai. Stage 1/2 mein right padding thi.

# ── 3.4 Stage 3 DPO Configuration ─────────────────────────────────────────────
stage3_config = DPOConfig(
    output_dir=f"{OUTPUT_ROOT}/stage3_logs",

    max_steps=STAGE3_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE3_LR,  # 0.00005 — sabse chota learning rate
                               # DPO very sensitive hai — bada LR se model collapse ho sakta hai

    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    beta=DPO_BETA,  # DPO loss formula: L = -log(sigmoid(beta * (r_chosen - r_rejected)))
                    # 0.1 = industry standard starting point

    max_length=MAX_SEQ_LENGTH,

    seed=SEED,
    remove_unused_columns=False,  # DPO trainer ko "prompt"/"chosen"/"rejected" sab chahiye
)

# ── 3.5 DPO Trainer Banao ──────────────────────────────────────────────────────
stage3_trainer = DPOTrainer(
    model=stage3_model,
    ref_model=None,   # Unsloth internally implicit reference handle karta hai
    processing_class=stage3_tokenizer,
    train_dataset=stage3_dataset,
    args=stage3_config,
)

# ── 3.6 Training Chalao — Time aur VRAM Khud Measure Karo ─────────────────────
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

stage3_start_time = time.time()
stage3_train_result = stage3_trainer.train()
torch.cuda.synchronize()

stage3_train_time = round(time.time() - stage3_start_time, 2)
stage3_peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
stage3_peak_reserved  = round(torch.cuda.max_memory_reserved()  / 1024**3, 3)

print("\nSTAGE 3 RESULTS")
print("Train time/sec:", stage3_train_time)
print("Peak allocated VRAM/GB:", stage3_peak_allocated)
print("Peak reserved VRAM/GB:", stage3_peak_reserved)

# ── 3.7 Final Test Answer (Generation Ke Liye Right Padding Wapas) ───────────
stage3_tokenizer.padding_side = "right"
FastLanguageModel.for_inference(stage3_model)

stage3_test_prompt = "### Instruction:\nExplain metformin in simple language.\n\n### Response:\n"
stage3_inputs = stage3_tokenizer(stage3_test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    stage3_output_ids = stage3_model.generate(
        **stage3_inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=stage3_tokenizer.eos_token_id,
        eos_token_id=stage3_tokenizer.eos_token_id,
    )

stage3_input_len = stage3_inputs["input_ids"].shape[-1]
stage3_answer = stage3_tokenizer.decode(
    stage3_output_ids[0][stage3_input_len:], skip_special_tokens=True
).strip()
print("\nFinal model test answer before merge:\n", stage3_answer)

# ── 3.8 Final Save + Merge ────────────────────────────────────────────────────
FastLanguageModel.for_training(stage3_model)

print("\nSaving Stage 3 DPO adapter...")
stage3_model.save_pretrained(STAGE3_ADAPTER_DIR)
stage3_tokenizer.save_pretrained(STAGE3_ADAPTER_DIR)
print("Stage 3 DPO adapter saved to:", STAGE3_ADAPTER_DIR)

print("\nMerging Stage 3 adapter with base model — FINAL production model...")
stage3_model.save_pretrained_merged(
    FINAL_MERGED_DIR,
    stage3_tokenizer,
    save_method="merged_16bit",
)
print("Final merged model saved to:", FINAL_MERGED_DIR)

# ── 3.9 Memory Free Karo ──────────────────────────────────────────────────────
del stage3_trainer
del stage3_model
gc.collect()
torch.cuda.empty_cache()
print("Stage 3 complete. Pipeline finished!")


## ✅ Final Output Paths — Sab Kuch Kahan Save Hua

Pipeline complete hone ke baad verify karo ke sab files properly save hui hain.


In [ ]:
# ============================================================
# Final Pipeline Summary — Sab Output Paths Print Karo
# ============================================================

print("\nPipeline completed successfully!")

print("\n" + "="*60)
print("STAGE 1 ARTIFACTS:")
print("Stage 1 adapter (LoRA only):", STAGE1_ADAPTER_DIR)
print("Stage 1 merged model (full):", STAGE1_MERGED_DIR)

print("\n" + "="*60)
print("STAGE 2 ARTIFACTS:")
print("Stage 2 adapter (LoRA only):", STAGE2_ADAPTER_DIR)
print("Stage 2 merged model (full):", STAGE2_MERGED_DIR)

print("\n" + "="*60)
print("STAGE 3 ARTIFACTS:")
print("Stage 3 DPO adapter (LoRA only):", STAGE3_ADAPTER_DIR)
print("Final merged model (production ready):", FINAL_MERGED_DIR)

print("\n" + "="*60)
print("Files in STAGE3_ADAPTER_DIR:")
print(os.listdir(STAGE3_ADAPTER_DIR))

print("\nFiles in FINAL_MERGED_DIR:")
print(os.listdir(FINAL_MERGED_DIR))


## 📊 Reference: Teenon Stages Ka Comparison

### Sequence (yeh order hamesha follow karo)
1. **Stage 1 — Non-Instruction FT**: raw PDF paragraphs → domain language seekhna
2. **Stage 2 — Instruction FT (SFT)**: Stage 1 ke merged model se shuru → Q&A format seekhna
3. **Stage 3 — Preference (DPO)**: Stage 2 ke merged model se shuru → behtar response choose karna

### Stage-wise Overview

| Stage | Data | Trainer | Model Kya Sikhta Hai |
|-------|------|---------|----------------------|
| **Stage 1 — Non-instruction** | Raw PDF paragraphs | `SFTTrainer` (packing=True) | Pharma domain language aur facts |
| **Stage 2 — Instruction SFT** | Instruction + Response JSONL | `SFTTrainer` (packing=False) | User instructions follow karna, Q&A format |
| **Stage 3 — DPO Preference** | Prompt + Chosen + Rejected | `DPOTrainer` | Kaunsa jawab behtar hai — preference alignment |

---

### Stage 1 vs Stage 2 — Key Differences

| Parameter | Stage 1 | Stage 2 | Kyun Faraq Hai |
|-----------|---------|---------|----------------|
| **Data format** | Raw paragraphs `{"text": "Metformin is..."}` | Formatted Q&A `### Instruction...### Response...` | Stage 1 language sikhata hai; Stage 2 format sikhata hai |
| **`packing`** | `True` | `False` | Raw paragraphs chhote hote hain packing best; instruction boundaries preserve karni hoti hain |
| **Learning rate** | `2e-4` (higher) | `1e-4` (lower) | Stage 2 mein careful update — purani knowledge overwrite na ho |
| **Starting model** | Base TinyLlama | Stage 1 merged model | Progressive training — aage se seekhte hain |

---

### Architecture: Konsa Library Kya Karta Hai

| Component | Source | Kaam |
|-----------|--------|------|
| `FastLanguageModel` | Unsloth | Fast 4-bit model loading + LoRA patching |
| `get_peft_model` style | Unsloth | Optimized LoRA training kernels |
| `SFTTrainer` | Hugging Face TRL | Next-token prediction training loop |
| `DPOTrainer` | Hugging Face TRL | Preference optimization training loop |
| `Unsloth patching` | Unsloth | TRL trainers ko memory-efficient banata hai |

---

> **Bottom Line:** Unsloth same fine-tuning karta hai jo Hugging Face karta hai, lekin:
> - **2x faster** — optimized CUDA kernels
> - **60% less VRAM** — gradient checkpointing + kernel fusion
> - **Same results** — accuracy mein koi faraq nahi
